# 07 — Park unimodality visualisation

This notebook imports the mean-CUSUM utilities from src/detection/stage1/park_mean_cusum.py, generates synthetic positive-valued multichannel data with a mean change, runs the Park mean-CUSUM pipeline (raw and smoothed), evaluates the unimodality/agreement checks, and visualises the results.

Created automatically by GitHub Copilot on request.

In [ ]:
# Imports: locate the repository from either root/ or notebooks/ working directory
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents)
        if (candidate / "src" / "detection" / "stage1" / "park_mean_cusum.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Cannot locate the ewma_covariate_shift_demo repository above "
        f"the notebook working directory: {cwd}. "
        "Launch Jupyter from the repository or its notebooks/ directory."
    )

# Import through the repository's canonical package path.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.detection.stage1.park_mean_cusum import (
    mean_cusum_curves,
    smooth_cusum_curve,
    check_cusum_unimodality,
    evaluate_mean_change,
)

print(f"Repository root: {repo_root}")
print("Imported src.detection.stage1.park_mean_cusum")


In [ ]:
# Synthetic positive-valued multichannel data generator
def make_synthetic_positive_data(n=300, p=12, change_at=120, channels_affected=None, magnitude=3.0, rng=None):
    rng = np.random.default_rng(rng)
    if channels_affected is None:
        channels_affected = np.arange(p // 2)
    data = rng.exponential(scale=1.0, size=(n, p))
    data[change_at:, channels_affected] += magnitude
    return data

# Example parameters and data
n, p = 300, 12
change_at = 120
data = make_synthetic_positive_data(n=n, p=p, change_at=change_at, magnitude=4.0, rng=42)
print('Synthetic data shape:', data.shape)


In [ ]:
# Run mean-CUSUM and evaluation
min_segment = 8
curves = mean_cusum_curves(data, min_segment=min_segment)

print('positions (len):', len(curves['positions']))
print('channels:', curves['channels'], 'observations:', curves['observations'])
print('b_max (raw):', curves['b_max'], 'b_avg (raw):', curves['b_avg'])
print('b_max_smoothed:', curves['b_max_smoothed'], 'b_avg_smoothed:', curves['b_avg_smoothed'])
print('agreement_gap_smoothed:', curves['agreement_gap_smoothed'])

decision = evaluate_mean_change(curves)
print('\nDecision summary:')
for k, v in decision.items():
    if k in ('maximum_check', 'average_check'):
        print('  {}: eligible={} unimodal={} A={}'.format(k, v['eligible'], v['unimodal'], v['A']))
    else:
        print(f'  {k}: {v}')


In [ ]:
# Visualisation: per-channel heatmap and aggregate curves
positions = np.array(curves['positions'])
maximum = np.array(curves['maximum'])
average = np.array(curves['average'])
maximum_sm = np.array(curves['maximum_smoothed'])
average_sm = np.array(curves['average_smoothed'])
by_channel = np.array(curves['by_channel'])

fig, axes = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2]})

# Top: heatmap of per-channel normalized CUSUM
ax = axes[0]
im = ax.imshow(by_channel, aspect='auto', origin='lower', cmap='viridis',
               extent=[positions[0], positions[-1], 0, by_channel.shape[0]])
ax.set_title('Channelwise normalized |CUSUM| (channels x positions)')
ax.set_ylabel('channel')
ax.set_xlabel('candidate position (b)')
fig.colorbar(im, ax=ax, label='normalized CUSUM')

# Bottom: aggregate curves and smoothed versions with peaks
ax = axes[1]
ax.plot(positions, maximum, label='maximum (raw)', color='C0', alpha=0.6)
ax.plot(positions, average, label='average (raw)', color='C1', alpha=0.6)
ax.plot(positions, maximum_sm, label='maximum (smoothed)', color='C0', linewidth=2)
ax.plot(positions, average_sm, label='average (smoothed)', color='C1', linewidth=2)

bmax = int(curves['b_max_smoothed'])
bavg = int(curves['b_avg_smoothed'])
ax.axvline(bmax, color='C0', linestyle='--', label=f'b_max_smoothed={bmax}')
ax.axvline(bavg, color='C1', linestyle='--', label=f'b_avg_smoothed={bavg}')

ax.set_title('Aggregated mean-CUSUM curves and smoothed peaks')
ax.set_xlabel('candidate position (b)')
ax.set_ylabel('aggregate normalized CUSUM')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


In [ ]:
# Unimodality sample points and details
max_check = decision['maximum_check']
avg_check = decision['average_check']

print('Maximum check:', max_check)
print()
print('Average check:', avg_check)

if max_check['eligible']:
    D1 = np.array(max_check['D1'])
    D2 = np.array(max_check['D2'])
    plt.figure(figsize=(9, 3))
    plt.plot(positions, maximum_sm, label='maximum_smoothed')
    plt.scatter(D1, maximum_sm[np.searchsorted(positions, D1)], label='D1 (left)', color='C2')
    plt.scatter(D2, maximum_sm[np.searchsorted(positions, D2)], label='D2 (right)', color='C3')
    plt.axvline(max_check['peak'], color='k', linestyle=':', alpha=0.6)
    plt.legend()
    plt.title('Maximum smoothed + sampled points for unimodality check')
    plt.show()
